# Converting coordinates from a csv file to a json file

In [ ]:
import json
import os
import pandas as pd
import numpy as np
import rasterio

from PIL import Image
from pyproj import Transformer   
from datetime import datetime    

## Import the raster file to get the metadata

We need the information from the raster as the json file needs to be in reference to the origin and transformation of the tif file.

In [ ]:
image_name = '2024-06_mimal_test_S2'

tif_base_dir = 'cookie-cutting/'
tif_path = os.path.join(tif_base_dir, image_name + '.tif')

with rasterio.open(tif_path) as raster:
    # Read the raster band
    imported_raster = raster.read(1)
    # Get the metadata of the raster
    imported_raster_meta = raster.meta
    # Get the raster transform parameters
    raster_transform = raster.transform

print("Shape of the raster (rows, columns):")
print(imported_raster.shape)
print("\n")

print("Raster metadata:")
print(imported_raster_meta)
print("\n")

print("Affine transformation parameters:")
print(raster_transform)

## Account for the png padding

As the png has padding, we need to figure out how much to add to the x and y coordinates to get the correct position for the labels in the png file.

This function will be different if there is padding on both sides and the top and bottom of the image. I've just accounted for padding on the left and top of the image by adding the difference in between the png size and the raster size to the x (difference in width) and y (difference in height) coordinates.

If the padding is consistent (e.g. tile_size - stride), we can just add that padding to the x and y coordinates. In that case, set width_diff and height_diff to the padding amount for the x and y axes respectively.

In [ ]:
Image.MAX_IMAGE_PIXELS = 933120000 # change to greater than the nnumber of pixels (only needed if there's a warning)

# Base directory for the png image
png_base_dir = 'cookie-cutting/'

# Load the png image
image = Image.open(f'{png_base_dir}{image_name}.png')

# Convert the image to a numpy array
pixel_data = np.array(image)

## Get image information

In [ ]:
# Get image information
png_width, png_height = image.size
print(f'Image size: ({png_width}, {png_height})')

# Calculate the different in the width and height of the image and the raster
width_diff = png_width - imported_raster.shape[1]
height_diff = png_height - imported_raster.shape[0]
print(f'Difference in width: {width_diff}')
print(f'Difference in height: {height_diff}')


# if using images with padded left and right (and top and bottom)
tile_size = 416
stride = 104
min_pad = tile_size - stride
# subtract the padding on the right and bottom from the differences
width_diff = width_diff - min_pad
height_diff = height_diff - min_pad
print(f'Difference in width after removing right padding: {width_diff}')
print(f'Difference in height after removing bottom padding: {height_diff}')

## Convert the locations in the csv file to the same projection as the tif file

As the locations are in WGS84, we need to convert them to the same projection as the tif file, and then convert them to pixel coordinates using the raster transformation from the metadata.

Edit - the tif coordinates are in WGS84, so we'll just go from EPSG4326 to EPSG4326.

In [ ]:
# Create the reprojection function
coord_transformer = Transformer.from_crs('epsg:4326', 'epsg:4326', always_xy=True)

# Test on sample set of coordinates
x,y = coord_transformer.transform(133.6153775, -13.7976017)
print(x,y)

# Check the pixel coordinates of the transformed coordinates
pixel_column, pixel_row = ~raster_transform * (x, y)
print(pixel_column, pixel_row)

## Read in the csv file of labelled coordinates

In [ ]:
# Specify the path to your CSV file
csv_base_dir = f'data/'
# csv_name = '2024 Rapid Waterhole Assessment_aligned.csv' # with health states
csv_name = '2024 Rapid Waterhole Assessment_aligned.csv'
csv_file_path = os.path.join(csv_base_dir, csv_name)

# Read the CSV file into a DataFrame
waterhole_labelled_df = pd.read_csv(csv_file_path)

print(f'Number of samples: {len(waterhole_labelled_df)}')
print('\n')

# Display the first few rows of the DataFrame
print(waterhole_labelled_df.head())

## Reproject the label coordinates to the same projection as the tif file

In [ ]:
x_proj, y_proj = coord_transformer.transform(
    waterhole_labelled_df['Longitude'].values, 
    waterhole_labelled_df['Latitude'].values
)

print(x_proj, y_proj)

## Convert the label coordinates to pixel coordinates

These pixel coordinates are in reference to the tif file, not the png yet (if it has padding).

In [ ]:
# Convert to pixel coordinates
pixel_coords = np.array([~raster_transform * (x, y) for x, y in zip(x_proj, y_proj)])
print(pixel_coords)

## Add the new columns to the dataframe

Here is where we add to the x and y coordinates to account for the padding in the png file.

In [ ]:
# Add the new columns to the dataframe
waterhole_labelled_df['x_proj'] = x_proj
waterhole_labelled_df['y_proj'] = y_proj
waterhole_labelled_df['pixel_col'] = pixel_coords[:, 0] + width_diff
waterhole_labelled_df['pixel_row'] = pixel_coords[:, 1] + height_diff

print(waterhole_labelled_df.head())

## Define the function to create the json file

Essentially we are just taking the pixel coordinates and the labels and creating a json file with the correct format.

In [ ]:
def csv_to_labelme(x, y, 
                   labels=None, 
                   image_path=None, 
                   image_height=None, 
                   image_width=None):
    """
    Convert coordinate columns from a DataFrame to LabelMe JSON format.
    
    Args:
        x (pd.Series): Series containing x/longitude coordinates
        y (pd.Series): Series containing y/latitude coordinates
        labels (pd.Series, optional): Series containing point labels. Defaults to None
        image_path (str, optional): Path to the corresponding image file. Defaults to None
        image_height (int, optional): Height of the image in pixels. Defaults to None
        image_width (int, optional): Width of the image in pixels. Defaults to None
        
    Returns:
        dict: LabelMe formatted JSON
    """
    # Validate inputs
    if len(x) != len(y):
        raise ValueError("x and y coordinates must have the same length")
    if labels is not None and len(labels) != len(x):
        raise ValueError("labels must have the same length as coordinates")
    
    # Initialize LabelMe JSON structure
    labelme_json = {
        "version": "5.0.1",
        "flags": {},
        "shapes": [],
        "imagePath": os.path.basename(image_path) if image_path else "",
        "imageData": None,  # LabelMe stores base64 image data here, but we'll leave it empty
        "imageHeight": image_height,
        "imageWidth": image_width
    }
    
    # Convert each point to LabelMe shape
    point_size = 5  # Size of the point representation in pixels
    
    for i in range(len(x)):
        # Skip if coordinates are NaN
        if pd.isna(x[i]) or pd.isna(y[i]):
            continue
            
        shape = {
            "label": str(labels.iloc[i]) if labels is not None else "point",
            "points": [
                [float(x[i]) - point_size, float(y[i]) - point_size],  # Top-left
                [float(x[i]) + point_size, float(y[i]) + point_size]   # Bottom-right
            ],
            "group_id": None,
            "shape_type": "rectangle",
            "flags": {}
        }
        
        labelme_json['shapes'].append(shape)
    
    # Add creation time
    labelme_json['timeStamp'] = datetime.now().isoformat()
    
    return labelme_json

## Function to save the json file

In [ ]:
# Convert the DataFrame to LabelMe JSON
def save_labelme_json(labelme_json, output_path):
    """Save the LabelMe JSON to file."""
    with open(output_path, 'w') as f:
        json.dump(labelme_json, f, indent=2)

## Run the csv to json function

The image path should be the name of the png file, and the output path of the save_labelme_json function should lead to where the png is saved.

In [ ]:
# Convert and save
labelme_json = csv_to_labelme(
    x=waterhole_labelled_df['pixel_col'],
    y=waterhole_labelled_df['pixel_row'],
    labels=waterhole_labelled_df['Class'],
    image_path=tif_name + '.png',
    image_height=png_height,
    image_width=png_width
)
    
# Save the LabelMe JSON file
save_labelme_json(labelme_json, f'{png_base_dir}/{tif_name}.json')
print(labelme_json)